In [9]:
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader

load_dotenv()

True

In [10]:
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader

load_dotenv()

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
files = reader.read()

documents = []
for file in files:
    doc = file.parse()
    
    documents.append(doc)

print(documents[0].keys())
print(documents[0]["filename"])
print(len(documents))

dict_keys(['content', 'filename'])
01-agentic-rag/lessons/01-intro.md
72


In [11]:
from minsearch import  Index
index = Index(
      text_fields=["content"],
    keyword_fields=["filename"],
)
index.fit(documents)
query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)
print(results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


In [12]:
from openai import OpenAI
from rag_helper import RAGBase

client = OpenAI()
rag = RAGBase(index=index, llm_client=client)

query = "How does the agentic loop keep calling the model until it stops?"
answer, usage = rag.rag(query)

print(answer)
print(usage.input_tokens)

It keeps calling the model inside a `while True` loop and checks each response for any `function_call` items.

- If the model returns a `function_call`, the code runs the tool, appends the tool output to the message history, and loops again.
- If the model returns only a normal `message` and no function calls, `has_function_calls` stays `False`, and the loop `break`s.

So the stop condition is: **no function calls in the latest response**.
7111


In [13]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))


295


In [14]:
chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

rag_chunks = RAGBase(index=chunk_index, llm_client=client)
answer_c, usage_c = rag_chunks.rag(query)
print(usage_c.input_tokens)


2294


In [16]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

# Search tool — type hint + docstring are required so toyaikit builds the schema
def search(query: str) -> list[dict]:
    """Search the course lessons for relevant content."""
    return chunk_index.search(query, num_results=5)

instructions = """
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
""".strip()

agent_tools = Tools()
agent_tools.add_tool(search)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

# Count how many times search was called
search_calls = sum(
    1 for m in result.all_messages
    if getattr(m, 'type', None) == 'function_call' and getattr(m, 'name', None) == 'search'
)
print("Search calls:", search_calls)

-> Response received


-> Response received


Search calls: 3
